<a href="https://colab.research.google.com/github/IlinoisZiwei/Hack-AI-thon-2026-Spring/blob/main/Hack_AI_thon_2026_Spring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
from google.colab import files
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from ipywidgets import interact, Text, Dropdown
from IPython.display import display

import json
import re
from pathlib import Path

In [3]:
uploaded = files.upload()

Saving Reviews_PROC.csv to Reviews_PROC.csv


In [9]:
# =========================
# 1. Load data
# =========================
reviews = pd.read_csv("Reviews_PROC.csv")

print("Shape:", reviews.shape)
print("Columns:", reviews.columns.tolist())
print(reviews.head())


# =========================
# 2. Keep useful columns
# =========================
reviews = reviews[
    [
        "eg_property_id",
        "acquisition_date",
        "lob",
        "rating",
        "review_title",
        "review_text",
    ]
].copy()


# =========================
# 3. Parse dates
# =========================
reviews["acquisition_date"] = pd.to_datetime(
    reviews["acquisition_date"], errors="coerce"
)


# =========================
# 4. Parse rating JSON string
#    Example:
#    '{"overall": 5.0, "service": 4.0, ...}'
# =========================
def safe_parse_rating(x):
    if pd.isna(x):
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}

reviews["rating_dict"] = reviews["rating"].apply(safe_parse_rating)
reviews["overall_rating"] = reviews["rating_dict"].apply(
    lambda d: d.get("overall", None)
)


# =========================
# 5. Fill missing title/text
# =========================
reviews["review_title"] = reviews["review_title"].fillna("").astype(str)
reviews["review_text"] = reviews["review_text"].fillna("").astype(str)


# =========================
# 6. Combine title + text
# =========================
def combine_review_text(title, text):
    title = title.strip()
    text = text.strip()

    if title and text:
        return f"{title}. {text}"
    elif title:
        return title
    else:
        return text

reviews["review_full_text"] = reviews.apply(
    lambda row: combine_review_text(row["review_title"], row["review_text"]),
    axis=1
)


# =========================
# 7. Clean review text
#    - lowercase
#    - remove extra spaces
#    - keep letters/numbers/basic punctuation
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)              # collapse spaces/newlines
    text = re.sub(r"[^\w\s.,!?'-]", " ", text)    # remove strange symbols
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews["review_text_clean"] = reviews["review_full_text"].apply(clean_text)


# =========================
# 8. Remove rows with no usable text
# =========================
reviews = reviews[reviews["review_text_clean"].str.len() > 0].copy()

print("\nAfter cleaning:")
print("Remaining rows:", len(reviews))


# =========================
# 9. Basic review-level features
# =========================
reviews["text_length_chars"] = reviews["review_text_clean"].str.len()
reviews["text_length_words"] = reviews["review_text_clean"].str.split().str.len()

print("\nSample cleaned reviews:")
print(
    reviews[
        [
            "eg_property_id",
            "acquisition_date",
            "overall_rating",
            "review_full_text",
            "review_text_clean",
        ]
    ].head(10)
)


# =========================
# 10. Group reviews by hotel
#     This is useful for later aspect analysis
# =========================
hotel_reviews = (
    reviews.groupby("eg_property_id")
    .agg(
        n_reviews=("review_text_clean", "count"),
        avg_overall_rating=("overall_rating", "mean"),
        latest_review_date=("acquisition_date", "max"),
        all_reviews=("review_text_clean", list),
    )
    .reset_index()
)

print("\nHotel-level grouped data:")
print(hotel_reviews.head())

Shape: (5999, 6)
Columns: ['eg_property_id', 'acquisition_date', 'lob', 'rating', 'review_title', 'review_text']
                                      eg_property_id acquisition_date    lob  \
0  9a0043fd4258a1286db1e253ca591662b3aac849da12d0...          2/10/23  HOTEL   
1  db38b19b897dbece3e34919c662b3fd66d23b615395d11...          2/14/23  HOTEL   
2  ff26cdda236b233f7c481f0e896814075ac6bed335e162...          2/14/23  HOTEL   
3  3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...           2/8/23  HOTEL   
4  823fb2499b4e37d99acb65e7198e75965d6496fd1c579f...           2/9/23  HOTEL   

                                              rating review_title  \
0  {"overall": 5.0, "roomcleanliness": 4.0, "serv...          NaN   
1  {"overall": 5.0, "roomcleanliness": 5.0, "serv...          NaN   
2  {"overall": 5.0, "roomcleanliness": 4.0, "serv...          NaN   
3  {"overall": 3.0, "roomcleanliness": 3.0, "serv...          NaN   
4  {"overall": 4.0, "roomcleanliness": 4.0, "serv...         

/tmp/ipykernel_15741/3466616261.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  reviews["acquisition_date"] = pd.to_datetime(



After cleaning:
Remaining rows: 4251

Sample cleaned reviews:
                                       eg_property_id acquisition_date  \
0   9a0043fd4258a1286db1e253ca591662b3aac849da12d0...       2023-02-10   
1   db38b19b897dbece3e34919c662b3fd66d23b615395d11...       2023-02-14   
2   ff26cdda236b233f7c481f0e896814075ac6bed335e162...       2023-02-14   
3   3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...       2023-02-08   
4   823fb2499b4e37d99acb65e7198e75965d6496fd1c579f...       2023-02-09   
5   fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...       2023-02-13   
7   fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...       2023-02-13   
8   3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...       2023-02-13   
9   3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...       2023-02-26   
10  7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...       2023-03-01   

    overall_rating                                   review_full_text  \
0              5.0                  mir hat der s

In [14]:
import pandas as pd
import json
import re
from pathlib import Path
from collections import Counter
from datetime import datetime

# =========================================================
# 1. Predefined hotel dimension system
#    Matches the doc's idea: hardware / service / surroundings / policy
# =========================================================
DIMENSIONS = {
    # ---------- Hardware facilities ----------
    "wifi_speed": {
        "category": "hardware",
        "keywords": ["wifi", "wi-fi", "internet", "signal", "connection", "speed"]
    },
    "soundproofing": {
        "category": "hardware",
        "keywords": ["soundproof", "soundproofing", "noise", "noisy", "quiet", "loud", "thin walls"]
    },
    "air_conditioning": {
        "category": "hardware",
        "keywords": ["air conditioning", "ac", "a/c", "heating", "heater", "temperature", "cold", "hot"]
    },
    "elevator_wait_time": {
        "category": "hardware",
        "keywords": ["elevator", "lift", "waiting for elevator", "slow elevator"]
    },
    "power_outlets": {
        "category": "hardware",
        "keywords": ["outlet", "plug", "power socket", "charging", "usb port"]
    },
    "water_pressure": {
        "category": "hardware",
        "keywords": ["water pressure", "pressure", "shower pressure"]
    },

    # ---------- Service experience ----------
    "front_desk_efficiency": {
        "category": "service",
        "keywords": ["front desk", "reception", "receptionist", "check in", "check-in", "check out", "check-out"]
    },
    "room_cleanliness": {
        "category": "service",
        "keywords": ["clean", "cleanliness", "dirty", "spotless", "filthy", "dusty", "stain", "smell"]
    },
    "restaurant_quality": {
        "category": "service",
        "keywords": ["restaurant", "dining", "food", "meal", "dinner", "lunch", "buffet"]
    },
    "breakfast_quality": {
        "category": "service",
        "keywords": ["breakfast", "coffee", "morning meal", "continental breakfast"]
    },
    "luggage_storage": {
        "category": "service",
        "keywords": ["luggage", "baggage", "bag storage", "stored our bags", "left our bags"]
    },
    "staff_friendliness": {
        "category": "service",
        "keywords": ["staff", "service", "friendly", "helpful", "rude", "manager", "employee"]
    },

    # ---------- Surroundings ----------
    "transport_access": {
        "category": "surroundings",
        "keywords": ["subway", "metro", "station", "bus", "transport", "airport", "train", "walking distance"]
    },
    "noise_level": {
        "category": "surroundings",
        "keywords": ["street noise", "traffic", "quiet area", "loud outside", "night noise", "noisy area"]
    },
    "nearby_dining": {
        "category": "surroundings",
        "keywords": ["restaurants nearby", "food nearby", "cafes nearby", "nearby dining", "nearby food"]
    },
    "walkable_attractions": {
        "category": "surroundings",
        "keywords": ["walk to", "close to attractions", "near attractions", "walkable", "walking distance"]
    },
    "location_convenience": {
        "category": "surroundings",
        "keywords": ["location", "convenient", "central", "close to everything", "good area"]
    },

    # ---------- Policy information ----------
    "checkout_time": {
        "category": "policy",
        "keywords": ["checkout", "check-out", "late checkout", "late check-out"]
    },
    "breakfast_hours": {
        "category": "policy",
        "keywords": ["breakfast hours", "breakfast time", "served until", "breakfast ended"]
    },
    "parking_fee": {
        "category": "policy",
        "keywords": ["parking fee", "parking", "garage", "car park", "parked", "valet"]
    },
    "pet_policy": {
        "category": "policy",
        "keywords": ["pet", "pets", "dog", "cat", "pet friendly", "pet-friendly"]
    },
}

ALL_DIMENSIONS = list(DIMENSIONS.keys())


# =========================================================
# 2. Optional mapping from dimension -> rating field
#    Uses structured sub-ratings when available
# =========================================================
dimension_rating_map = {
    "front_desk_efficiency": "checkin",
    "room_cleanliness": "roomcleanliness",
    "staff_friendliness": "service",
    "location_convenience": "location",
}

def numeric_rating_to_sentiment(score):
    if score is None:
        return None
    try:
        score = float(score)
    except Exception:
        return None

    if score == 0:
        return None
    if score >= 4:
        return "positive"
    if score <= 2:
        return "negative"
    return "neutral"


# =========================================================
# 3. Simple fallback text sentiment
#    This is the non-LLM MVP fallback.
#    You can later replace detect_dimensions_and_sentiment_llm(...)
# =========================================================
POS_WORDS = {
    "good", "great", "excellent", "amazing", "nice", "clean", "comfortable",
    "friendly", "helpful", "fast", "reliable", "convenient", "quiet",
    "spacious", "perfect", "love", "loved", "awesome", "smooth"
}

NEG_WORDS = {
    "bad", "poor", "terrible", "awful", "dirty", "slow", "broken", "noisy",
    "loud", "small", "uncomfortable", "rude", "unhelpful", "smelly", "stained",
    "worst", "issue", "problem", "delay", "crowded"
}

def split_sentences(text):
    if pd.isna(text):
        return []
    text = str(text).strip()
    if not text:
        return []
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def keyword_in_text(text, keyword):
    pattern = r"\b" + re.escape(keyword.lower()) + r"\b"
    return re.search(pattern, text.lower()) is not None

def matched_sentences_for_dimension(text, keywords):
    sentences = split_sentences(text)
    matched = []
    for sent in sentences:
        sent_lower = sent.lower()
        for kw in keywords:
            if keyword_in_text(sent_lower, kw):
                matched.append(sent)
                break
    return matched

# do dimension-specific sentiment, not just whole-review sentiment.
# Example: “Great location, but the Wi-Fi was terrible.”
# Whole-review sentiment is mixed.
# But this helper lets us isolate the Wi-Fi sentence and label Wi-Fi = negative.
def local_sentiment_from_sentences(sentences):
    if not sentences:
        return None

    pos_count = 0
    neg_count = 0

    for sent in sentences:
        tokens = re.findall(r"\b[\w'-]+\b", sent.lower())
        for t in tokens:
            if t in POS_WORDS:
                pos_count += 1
            if t in NEG_WORDS:
                neg_count += 1

    if pos_count > neg_count:
        return "positive"
    if neg_count > pos_count:
        return "negative"
    if pos_count == 0 and neg_count == 0:
        return None
    return "neutral"


# =========================================================
# 4. Optional LLM hook
#    Keep this function as a placeholder if you later want to
#    use GPT for the JSON extraction style from your doc.
# =========================================================
def detect_dimensions_and_sentiment_llm(review_text, review_date=None):
    """
    Placeholder for the LLM-based extraction described in your doc.

    Expected return format:
    [
        {
            "dimension": "wifi_speed",
            "sentiment": "negative",
            "evidence": "The wifi was really slow",
            "review_date": "2026-04-10"
        },
        ...
    ]

    For now, this pipeline uses the rule-based fallback below.
    """
    raise NotImplementedError("Add your OpenAI API call here if needed.")


# =========================================================
# 5. Rule-based extraction per review
#    Output format is intentionally aligned with your doc:
#    each review -> covered dimensions + sentiment + review date
# =========================================================
def extract_dimension_mentions_rule_based(row):
    review_text = str(row["review_text_clean"]).strip()
    review_date = row["acquisition_date"]
    rating_dict = row["rating_dict"]
    overall_rating = row.get("overall_rating", None)

    extracted = []

    for dimension, meta in DIMENSIONS.items():
        matched = matched_sentences_for_dimension(review_text, meta["keywords"])
        if not matched:
            continue

        sentiment = None
        sentiment_source = None
        numeric_rating_used = None

        # 1) structured rating if available
        rating_field = dimension_rating_map.get(dimension)
        if rating_field and isinstance(rating_dict, dict):
            numeric_rating_used = rating_dict.get(rating_field)
            sentiment = numeric_rating_to_sentiment(numeric_rating_used)
            if sentiment is not None:
                sentiment_source = f"rating:{rating_field}"

        # 2) local text sentiment
        if sentiment is None:
            sentiment = local_sentiment_from_sentences(matched)
            if sentiment is not None:
                sentiment_source = "local_text"

        # 3) overall rating fallback
        if sentiment is None:
            sentiment = numeric_rating_to_sentiment(overall_rating)
            if sentiment is not None:
                sentiment_source = "overall_rating"

        if sentiment is None:
            sentiment = "unknown"
            sentiment_source = "none"

        extracted.append({
            "eg_property_id": row["eg_property_id"],
            "dimension": dimension,
            "category": meta["category"],
            "sentiment": sentiment,
            "sentiment_source": sentiment_source,
            "numeric_rating_used": numeric_rating_used,
            "review_date": review_date,
            "evidence": " | ".join(matched[:2]),
            "review_title": row.get("review_title", ""),
            "review_text_clean": review_text,
        })

    return extracted


# =========================================================
# 6. Build dimension-level mentions dataset
# =========================================================
dimension_records = []

for _, row in reviews.iterrows():
    dimension_records.extend(extract_dimension_mentions_rule_based(row))

dimension_mentions = pd.DataFrame(dimension_records)

print("dimension_mentions rows:", len(dimension_mentions))
print(dimension_mentions.head(10))


# =========================================================
# 7. Build hotel profiles
#    Each hotel keeps every predefined dimension, even if mention_count = 0
# =========================================================

def sentiment_variance_score(sentiment_counts):
    """
    Higher means more mixed/conflicting.
    A simple, interpretable score for MVP.
    """
    pos = sentiment_counts.get("positive", 0)
    neu = sentiment_counts.get("neutral", 0)
    neg = sentiment_counts.get("negative", 0)
    total = pos + neu + neg

    if total == 0:
        return 0.0

    return min(pos, neg) / total

def dominant_sentiment(sentiment_counts):
    nonzero = {k: v for k, v in sentiment_counts.items() if v > 0}
    if not nonzero:
        return None
    return max(nonzero, key=nonzero.get)

# FIX: define all_property_ids first
all_property_ids = sorted(reviews["eg_property_id"].dropna().unique().tolist())

hotel_profiles = {}

for pid in all_property_ids:
    hotel_profiles[pid] = {}
    hotel_data = dimension_mentions[dimension_mentions["eg_property_id"] == pid]

    for dim in ALL_DIMENSIONS:
        dim_rows = hotel_data[hotel_data["dimension"] == dim].copy()

        if len(dim_rows) == 0:
            hotel_profiles[pid][dim] = {
                "category": DIMENSIONS[dim]["category"],
                "mention_count": 0,
                "last_mentioned": None,
                "dominant_sentiment": None,
                "sentiment_counts": {
                    "positive": 0,
                    "neutral": 0,
                    "negative": 0,
                    "unknown": 0,
                },
                "sentiment_variance": 0.0,
                "example_snippets": [],
            }
            continue

        counts = Counter(dim_rows["sentiment"].fillna("unknown"))
        clean_counts = {
            "positive": counts.get("positive", 0),
            "neutral": counts.get("neutral", 0),
            "negative": counts.get("negative", 0),
            "unknown": counts.get("unknown", 0),
        }

        last_mentioned = dim_rows["review_date"].max()
        snippets = (
            dim_rows["evidence"]
            .dropna()
            .astype(str)
            .loc[lambda s: s.str.len() > 0]
            .head(3)
            .tolist()
        )

        hotel_profiles[pid][dim] = {
            "category": DIMENSIONS[dim]["category"],
            "mention_count": int(len(dim_rows)),
            "last_mentioned": (
                last_mentioned.strftime("%Y-%m-%d")
                if pd.notna(last_mentioned) else None
            ),
            "dominant_sentiment": dominant_sentiment(clean_counts),
            "sentiment_counts": clean_counts,
            "sentiment_variance": round(sentiment_variance_score(clean_counts), 4),
            "example_snippets": snippets,
        }

# =========================================================
# 8. Convert hotel_profiles into a flat heatmap-style table
# =========================================================
heatmap_rows = []

for pid, profile in hotel_profiles.items():
    for dim, info in profile.items():
        heatmap_rows.append({
            "eg_property_id": pid,
            "dimension": dim,
            "category": info["category"],
            "mention_count": info["mention_count"],
            "last_mentioned": info["last_mentioned"],
            "dominant_sentiment": info["dominant_sentiment"],
            "positive_count": info["sentiment_counts"]["positive"],
            "neutral_count": info["sentiment_counts"]["neutral"],
            "negative_count": info["sentiment_counts"]["negative"],
            "unknown_count": info["sentiment_counts"]["unknown"],
            "sentiment_variance": info["sentiment_variance"],
            "example_snippets": " || ".join(info["example_snippets"]),
        })

hotel_profile_heatmap = pd.DataFrame(heatmap_rows)

print("\nHotel profile heatmap:")
print(hotel_profile_heatmap.head(20))


# =========================================================
# 9. Optional completeness score
#    Useful for the dashboard idea in your doc
# =========================================================
def compute_completeness_score(profile_dict):
    total_dims = len(profile_dict)
    covered = sum(1 for _, info in profile_dict.items() if info["mention_count"] > 0)
    if total_dims == 0:
        return 0.0
    return round(100 * covered / total_dims, 1)

completeness_rows = []
for pid, profile in hotel_profiles.items():
    completeness_rows.append({
        "eg_property_id": pid,
        "completeness_score": compute_completeness_score(profile),
        "covered_dimensions": sum(
            1 for _, info in profile.items() if info["mention_count"] > 0
        ),
        "total_dimensions": len(profile),
    })

hotel_completeness = pd.DataFrame(completeness_rows)
print("\nHotel completeness:")
print(hotel_completeness.head(13))

dimension_mentions rows: 6167
                                      eg_property_id              dimension  \
0  9a0043fd4258a1286db1e253ca591662b3aac849da12d0...     staff_friendliness   
1  ff26cdda236b233f7c481f0e896814075ac6bed335e162...     restaurant_quality   
2  ff26cdda236b233f7c481f0e896814075ac6bed335e162...   location_convenience   
3  3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...       room_cleanliness   
4  823fb2499b4e37d99acb65e7198e75965d6496fd1c579f...   location_convenience   
5  fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...  front_desk_efficiency   
6  fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...   walkable_attractions   
7  fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...       room_cleanliness   
8  fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...   location_convenience   
9  3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...       room_cleanliness   

       category sentiment        sentiment_source  numeric_rating_used  \
0       service  positive 